In [ ]:
# @markdown Run this cell to check this notebook's version.

current_version = "0.1.1"
notebook_name = "VLab4Mic_parameter_sweeps"  # Replace with the actual notebook name

# First of all check if ipywidgets is installed, if not through an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    
    try:
        import yaml
    except ImportError:
        raise ImportError("pyyaml is not installed. Please install it using 'pip install pyyaml' or 'conda install -c conda-forge pyyaml'.")

    import requests
    import base64

    # Create widgets
    github_repo = widgets.Text(
        value="",
        description="*GitHub Repository URL:",
        placeholder="e.g https://github.com/username/repository",
        layout=widgets.Layout(width="70%"),
        style={'description_width': '150px'},
    )

    github_token = widgets.Password(
        value="",
        description="*Personal Access Token:",
        placeholder="[Disabled] e.g. ghp_XXXXXXXXXXXXXXXXXXXX",
        disabled=True,
        layout=widgets.Layout(width="70%"),
        style={'description_width': '150px'},
    )

    test_button = widgets.Button(
        description="Test Connection",
        button_style='success',
    )

    output_area = widgets.HTML()

    # By default the repository is considered public
    if "repository_is_private" not in globals():
        repository_is_private = False
    elif repository_is_private:
        github_token.disabled = False
        github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"

    def on_test_button_clicked(b):
        global repository_is_private
        
        # Check that the GitHub repository URL is provided
        if not github_repo.value:
            output_area.value = "⚠️ Please provide a GitHub repository URL."
            return
        
        # Get the owner and repo name from the URL
        repo_url = github_repo.value.rstrip('/')
        github_owner, github_repo_name = repo_url.split('/')[-2:]
        
        # Online version checking file path
        version_file_path = "notebooks/notebook_latest_versions.yaml"
        version_url = f"https://api.github.com/repos/{github_owner}/{github_repo_name}/contents/{version_file_path}"

        # Do an initial request to check if the repository is public
        if not repository_is_private:
            version_response = requests.get(version_url)
            if version_response.status_code == 404:
                repository_is_private = True
                github_token.disabled = False
                github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"
                output_area.value = f"⚠️ We have detected that the repository might be private. Please provide a Personal Access Token and click 'Test Connection' again."
                return
        else:
            headers = {"Accept": "application/vnd.github.v3+json"}
            if not github_token.value:
                output_area.value = "⚠️ Personal Access Token is required for private repositories."
                return
            headers["Authorization"] = f"token {github_token.value}"

            version_response = requests.get(version_url, headers=headers)

        # Check the response status
        if version_response.status_code == 200:
            content = version_response.json()['content']
            decoded_content = base64.b64decode(content).decode('utf-8')
            config = yaml.safe_load(decoded_content)
            latest_version = config.get(notebook_name, "")

            output_area.value = (f"<b>Notebook version:</b> `{current_version}`<br>"
                                f"<b>Latest version available:</b> `{latest_version}`<br>")

            if latest_version == "":
                output_area.value += "⚠️ This notebook is not listed in the version file.<br>"
            elif current_version == latest_version:
                output_area.value += "✅ This notebook is up-to-date.<br>"
            else:
                output_area.value += f"⚠️ A new version of this notebook is available."
        else:
            output_area.value += "⚠️ Could not retrieve the version file.<br>"

    test_button.on_click(on_test_button_clicked)
    # Widget layout
    widget_box = widgets.VBox([github_repo, github_token, test_button, output_area])
    display(widget_box)
except ImportError:
    display(widgets.HTML('To check the notebook version, please go to the <a href="../Welcome.ipynb" target="_blank" style="color: #0066cc; text-decoration: underline;">Welcome notebook</a>.'))

# 🔬 VLab4Mic: Parameter Sweeps for Microscopy Simulation

## What is VLab4Mic?
VLab4Mic is a comprehensive, modular package that enables researchers to:
- **Model labeling strategies** of macromolecular complexes
- **Simulate image acquisitions** under diverse microscopy modalities  
- **Find optimal parameters** for feature recovery across imaging conditions

Perfect for method developers, microscopy core facilities, and researchers seeking to optimize their imaging workflows before experimental work.

## 🧪 Parameter Sweep Workflow Overview

This notebook guides you through **systematic parameter exploration** for microscopy simulations. You'll create comprehensive datasets by testing multiple parameter combinations across your chosen configurations.

### 🚀 What You'll Accomplish:
- **Generate extensive simulation datasets** from parameter combinations
- **Create reference images** for comparative analysis  
- **Analyze simulations** against your reference standards
- **Optimize experimental parameters** before real laboratory work

---

### Step-by-Step Workflow:

1. **🏗️ Choose a Structure** - Select your macromolecular complex of interest
2. **🎯 Configure Probes** - Set up labeling strategies and structural integrity modeling
3. **📡 Select Modalities** - Choose imaging techniques (SMLM, confocal, widefield)
4. **⚙️ Define Parameter Sweeps** - Set ranges for systematic testing:
   - **Probe parameters:** efficiency, distance to epitope
   - **Structural Integrity modeling:** fraction, small/large cluster thresholds  
   - **Sample parameters:** particle count, orientations
   - **Acquisition settings:** exposure time, noise levels
5. **📊 Execute & Analyze** - Generate comprehensive datasets and comparisons

---

## 🔄 Understanding Parameter Sweeps

A **parameter sweep** systematically explores the parameter space by running simulations for **every possible combination** of your selected parameter values across:
- Molecular structures + Probe configurations + Imaging modalities + Structural Integrity settings

### ⚠️ Key Considerations:
- **📈 Exponential scaling:** More parameters = exponentially more simulations
- **🎛️ Parameter interactions:** Probe modifiers affect all selected probes  
- **🧬 Structural Integrity modeling:** Parameters work together to simulate realistic structural defects
- **💻 Computational cost:** Balance thoroughness with available resources

### 💡 **Pro Tip:** Start with small parameter ranges to test your workflow, then expand for comprehensive studies.

---

**Ready to begin?** Run each cell below to display interactive widgets and configure your parameter sweep.

# 📦 Import Dependencies

Run this cell to load all necessary VLab4Mic modules and initialize the parameter sweep system.

In [ ]:
#@title  Import VLab4Mic modules
try:
    import vlab4mic
except ImportError:
    !pip install vlab4mic
    import vlab4mic

try:
    import vlab4micjupyter
except ImportError:
    !pip install vlab4micjupyter
    import vlab4micjupyter


from vlab4micjupyter import sweep_parameters_widgets
from vlab4mic.sweep_generator import sweep_generator
from vlab4mic.utils.io.yaml_functions import load_yaml
from vlab4mic.analysis import sweep

# Initialize the sweep generator
print("🔄 Initializing VLab4Mic parameter sweep system...")
sweep_generator = sweep_generator()

# Verify parameter groups are loaded
print("\n📋 Available parameter groups:")
for group_name, params in sweep_generator.param_settings.items():
    print(f"  • {group_name}: {', '.join(params.keys())}")

print("\n🎉 System ready! Proceed to structure selection below.")

In [ ]:
#@title Optional: Run this cell to connect your Google Drive to the notebook session in Colab.
# check if running notebook in google colab
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f'Google Drive was not connected ({e}). Continuing without it.')

# 🏗️ Step 1: Choose Your Molecular Structure

Select the macromolecular complex you want to study. Parameter sweeps currently support **one structure at a time** to ensure focused analysis.

### 📚 Available Structures:
The widget below displays structures from the VLab4Mic database, including viral particles, nuclear pore complexes, and other well-characterized molecular assemblies.

**Instructions:** Run the cell below and select your structure from the dropdown menu.

In [ ]:
#@title Select structure
select_structres = sweep_parameters_widgets.select_structure(sweep_generator)
select_structres.show()

# 🎯 Step 2: Configure Probes and Modalities

Once you've selected a structure, choose your **labeling strategies** (probes) and **imaging modalities**. Multiple selections allow comprehensive cross-comparisons in your parameter sweep.

### 🔬 Selection Process:
1. **Probes:** Choose from available labeling strategies (antibodies, fluorescent proteins, etc.)
2. **Modalities:** Select imaging techniques (SMLM, confocal, widefield, etc.)

### 📝 Selection Instructions:
- **Multiple selection methods:**
  - **Click and drag:** Hold and slide to highlight multiple consecutive options
  - **Ctrl/Cmd + Click:** Hold Ctrl (Windows/Linux) or Cmd (Mac) and click individual options
- **Confirm selection:** Click the "Select" button to accept your highlighted choices

### 📖 Additional Resources:
For detailed information about available probes and modalities, consult our [documentation](https://website-vlab4mic.pages.dev/parameters/).

### ✅ Quick Start:
You can proceed with default selections and add parameter modifications in the next step. However, the following step allows you to customize parameter ranges for systematic exploration.


In [ ]:
#@title Select probes and imaging modalities
select_structres = sweep_parameters_widgets.select_probes_and_mods(sweep_generator)
select_structres.show()

# ⚙️ Step 3: Configure Parameter Sweep Ranges

Define the parameter ranges for systematic exploration. Each module (probe, structural integrity modeling, sample, acquisition) offers specific parameters that modify your configurations.


### 🎛️ Parameter Categories:

| Category | Parameters | Purpose |
|----------|------------|---------|
| **🎯 Probe** | Labeling Efficiency, Probe Distance to Epitope | Optimize labeling strategies |
| **🧬 Structural Integrity** | Structural Integrity Fraction, Small Cluster, Large Cluster | Model realistic structural defects |
| **📦 Sample** | Random Orientations, Number of Particles | Control sample properties |
| **📸 Acquisition** | Exposure Time | Optimize imaging conditions |

### 📊 Configuration Process:
1. **☑️ Select parameters:** Check the box for parameters you want to vary
2. **📏 Set ranges:** Adjust the range sliders (min, max values)  
3. **🔢 Define intervals:** Set the number of values to test within each range
4. **✅ Confirm:** Click "Select parameters for sweep" to apply your settings

### 🧮 **Computational Considerations:**
- **Total simulations** = Product of all interval counts across selected parameters
- **Example:** 3 parameters × 4 intervals each = 4³ = 64 simulations
- **Recommendation:** Start small (2-3 intervals) for testing, expand for production runs

### IMPORTANT NOTE: if using list of values, make sure "Use list of values" is also selected. Otherwise, the values for parameter sweep will be generated by considering the range and stepsize.


### 🔧 Boolean Parameters:
For True/False parameters, choose:
- **True:** Only test the enabled state
- **False:** Only test the disabled state  
- **Both:** Test both enabled and disabled states

#### Note: Structural Integrity Small Cluster is used to group epitopes, while Structural Integrity Large Cluster determines which clusters are considered neighbours in the removal phase.


In [ ]:
#@title Set up parameters values for sweep
select_parameters_values = sweep_parameters_widgets.add_parameters_values(sweep_generator)
select_parameters_values.show()

# 📸 Step 4: Generate Reference Image

Create a **gold standard reference image** for comparative analysis against your parameter sweep results. This reference serves as the baseline for evaluating parameter performance.

### 🎯 Default Reference Configuration:
- **Structure:** Same structure selected for parameter sweep
- **Probe:** NHS ester (standard labeling strategy)
- **Modality:** Reference modality optimized for comparison

### 🛠️ Advaced Parameters:
- **Upload image for reference:** Select an image to use as image reference
- **(Opitonal) Image mask for reference:** Provide an image mask for the reference image. If not image is provided, the analysis will use the whole image.
- **Pixel size (nm):** Specify the pixel size for the image reference

### 📋 Reference Purpose:
The reference image enables:
- **Performance benchmarking** across parameter combinations
- **Quantitative comparison** using image similarity metrics
- **Optimization guidance** for parameter selection

### 🔧 Instructions:
Click **"Set Reference"** below to generate your baseline image using the default optimal parameters.

### 📖 Learn More:
For detailed information about reference modalities and their properties, check our [documentation](https://website-vlab4mic.pages.dev/methods/).


In [ ]:
#@title Set up reference image
set_reference = sweep_parameters_widgets.set_reference(sweep_generator)
set_reference.show()

# 📊 Step 5: Execute Analysis

Perform comprehensive analysis comparing each simulated image from your parameter sweep against the reference standard. This quantitative comparison reveals optimal parameter combinations.

### 🔍 Analysis Methodology:
- **Metric:** SSIM (Structural Similarity Index Measure)
- **Comparison:** One-to-one comparison of each parameter combination vs. reference
- **Output:** Quantitative similarity scores for parameter optimization

### 📈 What You'll Get:
- **Performance rankings** of parameter combinations
- **Similarity scores** for quantitative comparison
- **Optimization insights** for experimental design
- **Visual comparisons** between simulated and reference images

### 🧮 SSIM Metric:
SSIM measures structural similarity considering:
- **Luminance:** Brightness comparison
- **Contrast:** Local variations  
- **Structure:** Spatial correlations
- **Range:** 0 (completely different) to 1 (identical)

### 📚 Advanced Metrics:
Currently supports SSIM. For additional analysis metrics and advanced comparison methods, refer to our [documentation](https://website-vlab4mic.pages.dev/methods/).

### 🚀 Ready to Analyze:
Click the button below to execute the parameter sweep analysis.


In [ ]:
#@title Run parameter sweep
analyse_sweep = sweep_parameters_widgets.analyse_sweep(sweep_generator)
analyse_sweep.show()

# 🎯 Workflow Complete!

Congratulations! You've successfully configured and executed a comprehensive parameter sweep analysis with VLab4Mic.

## 📊 What You've Accomplished:
- ✅ **Structure Selection:** Configured your macromolecular complex
- ✅ **Probe & Modality Setup:** Selected labeling strategies and imaging techniques  
- ✅ **Parameter Configuration:** Defined systematic parameter exploration ranges
- ✅ **Reference Generation:** Created baseline images for comparison
- ✅ **Quantitative Analysis:** Executed SSIM-based parameter optimization

## 🚀 Next Steps:

### 📈 **Analyze Your Results:**
- Review the similarity scores to identify optimal parameter combinations
- Examine which parameter ranges yielded the best performance
- Consider the trade-offs between different parameter settings

### 🔬 **Apply to Your Research:**
- Use the optimal parameters identified for your experimental work
- Consider running focused sweeps around the best-performing parameter ranges
- Validate findings with pilot experiments

### 📚 **Further Exploration:**
- Explore additional structures and modalities in VLab4Mic
- Try different reference configurations for alternative comparisons
- Investigate advanced analysis metrics beyond SSIM

### 💾 **Save Your Work:**
- Export parameter configurations for reproducibility
- Document optimal settings for future reference
- Share findings with your research team

---

## 🤝 **Need Help?**
- 📖 **Documentation:** Check our comprehensive [documentation](https://website-vlab4mic.pages.dev/)
- 💬 **Community:** Join our user community for discussions
- 🐛 **Issues:** Report bugs or request features on GitHub
- 📧 **Contact:** Reach out for technical support

**Happy simulating with VLab4Mic!** 🔬✨
